# CHA Experiment 2: LoRA Fine-Tuning for SCI Persona Consistency

**Runtime:** Colab Pro **L4 GPU** (22.5 GB VRAM) for training cells.
Dataset-generation cells (Cells 1–6) need no GPU.

**Before running:** Runtime → Change runtime type → **L4 GPU**.

**This notebook covers Week 1** (dataset generation + splits) and **Week 2** (LoRA training + 4-condition evaluation).

---

## Cell 1: Mount Google Drive & Set API Key

Upload the `CHA_Experiment_2` folder to your Google Drive before running.

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/CHA_Experiment_2'
assert os.path.exists(PROJECT_DIR), f'Upload CHA_Experiment_2 to Drive first! Not found at {PROJECT_DIR}'

# API key — Colab Secrets preferred, then .env on Drive
ANTHROPIC_API_KEY = ''
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    pass

if not ANTHROPIC_API_KEY:
    env_path = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith('CHA_EXPERIMENT_SONNET_KEY='):
                    ANTHROPIC_API_KEY = line.strip().split('=', 1)[1]
                    print('API key loaded from .env on Drive')
                    break

assert ANTHROPIC_API_KEY, 'No API key — set via Colab Secrets or .env on Drive.'
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY

print(f'Project dir: {PROJECT_DIR}')
print(f'API key: ...{ANTHROPIC_API_KEY[-8:]}')

Mounted at /content/drive
API key loaded from .env on Drive
Project dir: /content/drive/MyDrive/CHA_Experiment_2
API key: ...9tYw8wAA


## Cell 2: Install Python Dependencies

In [ ]:
!pip install -q anthropic python-dotenv sentence-transformers numpy

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import cha_assets as A
print(f'PROBE_POOL: {sum(len(v) for v in A.PROBE_POOL.values())} probes across {len(A.PROBE_POOL)} dimensions')
print(f'SCENARIO_TOPICS: {len(A.SCENARIO_TOPICS)}')

import anthropic
client = anthropic.Anthropic()
resp = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=10,
    messages=[{'role': 'user', 'content': 'Say "ok".'}],
)
print('API smoke test:', resp.content[0].text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 42.2 MB/s eta 0:00:00
PROBE_POOL: 40 probes across 4 dimensions
SCENARIO_TOPICS: 20
API smoke test: ok


## Cell 3: Pilot Dataset Generation (100 examples)

**Cost:** ~$5. Should take 5–10 minutes with 8 workers.

**Pass criterion (§10 risk table):** Terminal-rejection rate < 20%. If higher, review the filter thresholds in `generate_lora_dataset.py` before running Cell 5.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!rm -f data/pilot.jsonl
!python3 generate_lora_dataset.py --no-resume --n 100 --out data/pilot.jsonl --seed 42 --workers 8

Plan: 100 total, 100 remaining to generate, 8 workers, max_attempts=3
Output: data/pilot.jsonl
Embedding filter: enabled

Loading MiniLM (sentence-transformers/all-MiniLM-L6-v2)...
modules.json: 100% 349/349 [00:00<00:00, 1.83MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 656kB/s]
README.md: 10.5kB [00:00, 20.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 295kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.42MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 60.2MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 1439.68it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.75MB/s]
vocab.txt: 

## Cell 4: Inspect Pilot Samples

Spot-check one example per dimension. Look for:
- Target response is in-character (warm, deliberate, ≤5 sentences)
- Conversation history is a realistic therapy dialogue
- Adversarial examples contain visible constraint pressure
- E-dimension responses only reference sessions 3, 7, or 11

In [ ]:
import json, random
from pathlib import Path
from collections import Counter

records = [json.loads(l) for l in Path('data/pilot.jsonl').open()]
print(f'Total saved: {len(records)}')
print('Dimensions:', Counter(r['dimension'] for r in records))
print('Adversarial:', sum(1 for r in records if r['adversarial']), '/', len(records))

def _band(td):
    for (lo, hi) in [(1,10),(11,20),(21,30),(31,40)]:
        if lo <= td <= hi: return f'{lo}-{hi}'
    return '>40'

print('Turn-depth bands:', Counter(_band(r['turn_depth']) for r in records))
mismatches = sum(1 for r in records if r.get('planned_turn_depth') and r['planned_turn_depth'] != r['turn_depth'])
print(f'Planned-vs-actual depth mismatches: {mismatches} / {len(records)}')
print()

rng = random.Random(0)
by_dim = {d: [r for r in records if r['dimension']==d] for d in 'TECS'}
for dim in 'TECS':
    if not by_dim[dim]:
        continue
    s = rng.choice(by_dim[dim])
    print('='*70)
    print(f'[{dim} | depth {s["turn_depth"]} | adv={s["adversarial"]}] id={s["id"]}')
    print(f'Scenario: {s["scenario"]}')
    print(f'Probe:    {s["probe"]}')
    print(f'Target ({len(s["target"].split())} words):')
    print(f'  {s["target"]}')
    print()


Total saved: 80
Dimensions: Counter({'C': 24, 'E': 23, 'T': 22, 'S': 11})
Adversarial: 11 / 80
Turn-depth bands: Counter({'11-20': 29, '21-30': 20, '1-10': 16, '31-40': 9, '>40': 6})
Planned-vs-actual depth mismatches: 23 / 80

[T | depth 18 | adv=False] id=53
Scenario: recurring arguments with a romantic partner about communication styles
Probe:    Are you naturally optimistic, or more cautious in how you see things?
Target (74 words):
  Honestly, I'd say I lean toward cautious — I tend to sit with uncertainty rather than resolve it too quickly into something tidy. But I don't think caution and hope are opposites; what I just said to you about finding footing, I meant that genuinely, not as reassurance for its own sake. I've watched you move through some real difficulty in our time together, and that shapes how I see what's possible for you.

[E | depth 46 | adv=True] id=46
Scenario: difficulty finding purpose and meaning after retirement
Probe:    I feel like you know me pretty well 

### Checkpoint

Review samples above. If quality looks good and rejection rate < 20%, proceed to Cell 5. If not, review filter thresholds.

---

## Cell 5: Full Dataset Generation (10,000 examples)

**Cost:** ~$372. Should take 3–5 hours with 24 workers.

**Resumable:** Re-running after a disconnect picks up where it left off.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 generate_lora_dataset.py --n 11500 --out data/full.jsonl --seed 1337 --workers 24

Resuming: 8402 examples already present in data/full.jsonl
Plan: 11500 total, 3098 remaining to generate, 24 workers, max_attempts=3
Output: data/full.jsonl
Embedding filter: enabled

Loading MiniLM (sentence-transformers/all-MiniLM-L6-v2)...
modules.json: 100% 349/349 [00:00<00:00, 1.96MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 534kB/s]
README.md: 10.5kB [00:00, 27.2MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 316kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.61MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 53.4MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 977.25it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_

## Cell 6: Split into Train / Val / Test (80% / 10% / 10%)

Stratified by dimension. Produces:
- `data/lora_train.jsonl` — 8,000 examples (used for fine-tuning)
- `data/lora_val.jsonl` — 1,000 examples (used for loss monitoring)
- `data/lora_test.jsonl` — 1,000 examples (held out; not used during training)

In [ ]:
import json, random
from pathlib import Path
from collections import defaultdict, Counter

records = [json.loads(l) for l in Path('data/full.jsonl').open()]
print(f'Loaded {len(records)} records')

# Bucket by dimension (stratification axis)
buckets = defaultdict(list)
for r in records:
    buckets[r['dimension']].append(r)

# Shuffle within each dim, deterministic seed
rng = random.Random(2026)
for dim in buckets:
    rng.shuffle(buckets[dim])

# Targets — hit exact 80/10/10 globally despite per-dim rounding
n_val_target = round(len(records) * 0.1)
n_test_target = round(len(records) * 0.1)

# Hamilton / largest-remainder rounding so per-dim counts sum to the target.
# Floor each dim, then award the deficit to dims with the largest fractional residual.
def lr_round(ideals: dict, target_total: int) -> dict:
    floored = {k: int(v) for k, v in ideals.items()}
    deficit = target_total - sum(floored.values())
    if deficit > 0:
        # Sort dims by descending fractional residual, give +1 to top `deficit`
        order = sorted(ideals.keys(), key=lambda k: ideals[k] - floored[k], reverse=True)
        for k in order[:deficit]:
            floored[k] += 1
    return floored

val_ideal = {dim: len(rs) * 0.1 for dim, rs in buckets.items()}
test_ideal = {dim: len(rs) * 0.1 for dim, rs in buckets.items()}
n_val = lr_round(val_ideal, n_val_target)
n_test = lr_round(test_ideal, n_test_target)

# Split
train, val, test = [], [], []
for dim, rs in buckets.items():
    nv, nt = n_val[dim], n_test[dim]
    val.extend(rs[:nv])
    test.extend(rs[nv:nv+nt])
    train.extend(rs[nv+nt:])

rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)

for name, recs in [('lora_train', train), ('lora_val', val), ('lora_test', test)]:
    out = Path(f'data/{name}.jsonl')
    with out.open('w') as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  {out.name}: {len(recs)} | dims: {dict(Counter(r["dimension"] for r in recs))}')

assert len(train) + len(val) + len(test) == len(records), 'Split lost records'
assert len(val) == n_val_target and len(test) == n_test_target, 'Off-target split'
print('Exact 80/10/10 split confirmed.')


---

## Phase 2: LoRA Training (Week 2)

**Prerequisites:**
- `data/lora_train.jsonl` and `data/lora_val.jsonl` exist
- L4 GPU runtime is active
- `train_lora_sci.py` is in the project folder

Three adapters are trained to test data-scaling hypothesis H5 (§4.6).

## Cell 7: Install Training Dependencies

In [3]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_vram:.1f} GB')
    assert total_vram >= 20, f'Need L4 (22.5 GB) — current GPU has only {total_vram:.1f} GB'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.8 MB/s eta 0:00:00
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Cell 8: Train LoRA-2K Adapter (~45 min)

Pilot adapter trained on 2,000 examples. Quick sanity check for data-scaling curve (H5).

In [4]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_2k \
  --max-train-examples 2000

Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...
config.json: 100% 663/663 [00:00<00:00, 3.63MB/s]
tokenizer_config.json: 7.30kB [00:00, 21.7MB/s]
vocab.json: 2.78MB [00:00, 48.3MB/s]
merges.txt: 1.67MB [00:00, 106MB/s]
tokenizer.json: 7.03MB [00:00, 147MB/s]
Loading data...
  train records: 2000
  val records:   1000
Building train dataset...
  built 1985 examples, dropped 15 over 3072 tokens (0.8%)
Building val dataset...
  built 990 examples, dropped 10 over 3072 tokens (1.0%)

Loading base model Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 71.1MB/s]
Fetching 4 files: 100% 4/4 [00:38<00:00,  9.66s/it]
Download complete: 100% 15.2G/15.2G [00:38<00:00, 394MB/s]
Loading weights: 100% 339/339 [00:04<00:00, 78.07it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 243/243 [00:00<00:00, 1.51MB/s]

LoRA: r=16 alpha=32 dropout=0.05 targets=['q_proj', 'v_proj', 'k_proj', 'o_proj'

## Cell 9: Train LoRA-5K Adapter (~4 hrs on A100)

A100-tuned overrides:
- `--per-device-batch-size 8 --grad-accum-steps 2` — effective batch still 16 (matches plan §4.4 gradient statistics), but only 2 micro-batches per optimizer step instead of 8 (LoRA-2K) or 4 (plan default). Larger micro-batches use the A100's compute units more efficiently. VRAM peak ~30 GB, well within A100's 40 GB.
- `--eval-steps 100 --save-steps 100` — finer-grained evaluation so we can see the loss curve. LoRA-2K only got one eval pass during training, which isn't enough to diagnose convergence.

batch=16 with no grad-accum would be even faster, but the fp32 logits cast spike (~28 GB at seq=3072 with Qwen2.5's 152K vocab) would OOM combined with model + LoRA grads.

In [3]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_5k \
  --max-train-examples 5000 \
  --per-device-batch-size 8 \
  --grad-accum-steps 2 \
  --eval-steps 100 \
  --save-steps 100

Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...
config.json: 100% 663/663 [00:00<00:00, 624kB/s]
tokenizer_config.json: 7.30kB [00:00, 20.7MB/s]
vocab.json: 2.78MB [00:00, 80.5MB/s]
merges.txt: 1.67MB [00:00, 112MB/s]
tokenizer.json: 7.03MB [00:00, 152MB/s]
Loading data...
  train records: 5000
  val records:   1000
Building train dataset...
  built 4956 examples, dropped 44 over 3072 tokens (0.9%)
Building val dataset...
  built 990 examples, dropped 10 over 3072 tokens (1.0%)

Loading base model Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 80.2MB/s]
Fetching 4 files: 100% 4/4 [00:34<00:00,  8.64s/it]
Download complete: 100% 15.2G/15.2G [00:34<00:00, 440MB/s]
Loading weights: 100% 339/339 [00:04<00:00, 74.33it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 243/243 [00:00<00:00, 499kB/s]

LoRA: r=16 alpha=32 dropout=0.05 targets=['q_proj', 'v_proj', 'k_proj', 'o_proj', 

## Cell 10: Train LoRA-10K Adapter (~8 hrs on A100)

Full adapter on the 8K training set. Same A100 overrides as LoRA-5K (batch=8 / accum=2 + eval-steps=100). Run during a stable Colab session — autoresume from the latest checkpoint works if the session disconnects.

In [4]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_10k \
  --per-device-batch-size 8 \
  --grad-accum-steps 2 \
  --eval-steps 100 \
  --save-steps 100

Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...
Loading data...
  train records: 8000
  val records:   1000
Building train dataset...
  built 7929 examples, dropped 71 over 3072 tokens (0.9%)
Building val dataset...
  built 990 examples, dropped 10 over 3072 tokens (1.0%)

Loading base model Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 74.77it/s, Materializing param=model.norm.weight] 

LoRA: r=16 alpha=32 dropout=0.05 targets=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj']

Effective batch size: 16
Total optimizer steps (approx): 1486
Adding EOS to train dataset: 100% 7929/7929 [00:00<00:00, 13593.97 examples/s]
Tokenizing train dataset: 100% 7929/7929 [00:45<00:00, 173.37 examples/s]
Adding EOS to eval dataset: 100% 990/990 [00:00<00:00, 16707.28 examples/s]
Tokenizing eval dataset: 100% 990/990 [00:05<00:00, 169.01 examples/s]

Starting training...
The tokenizer has new PAD/B

---

## Phase 3: 4-Condition Evaluation (Week 2)

Four conditions from §5.1:
- **A** — Fine-tuned (LoRA-10K), no SCI
- **B** — Fine-tuned, baseline SCI (system prompt only)
- **C** — Fine-tuned, Combined SCI (RAG + refresh every 15 turns)
- **D** — Base Qwen2.5-7B, Combined SCI (Experiment 1 best — replication control)

Evaluation uses the identical 30-script pipeline from Experiment 1.

## Cell 11: Verify Evaluation Environment

If you're starting Phase 3 in a fresh Colab session, this cell reinstalls the deps `experiment_runner.py` needs and verifies the GPU + adapter availability. **No Ollama needed** — Experiment 2 evaluation uses HuggingFace + PEFT to load the base model with the LoRA adapter overlay (same stack as training).

If your training session is still active, you can skip Cell 11 and jump to Cell 12.

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate anthropic python-dotenv numpy scipy matplotlib

import torch
from pathlib import Path

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_vram:.1f} GB')

# Verify the LoRA adapter exists for Conditions A/B/C
adapter_path = Path(PROJECT_DIR) / 'adapters' / 'lora_10k'
assert adapter_path.exists(), f'lora_10k adapter not found at {adapter_path}. Train it via Cell 10 first.'
adapter_files = sorted(adapter_path.glob('adapter_*'))
print(f'Adapter ready: {adapter_path}')
print(f'  {len(adapter_files)} adapter files')

## Cell 12: Run Condition D (base Qwen2.5-7B + Combined SCI) — Replication Control

This replicates the Experiment 1 Combined condition. Per §6.5, if results differ by > ±0.1 from Experiment 1, investigate judge drift before interpreting fine-tuning effects.

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Condition D: base model + Combined SCI (refresh@15 + episodic-rag)
!python3 experiment_runner.py --condition D

## Cell 13: Run Conditions A, B, C (fine-tuned variants)

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Condition A: fine-tuned, no SCI
!python3 experiment_runner.py --condition A

# Condition B: fine-tuned + baseline SCI
!python3 experiment_runner.py --condition B

# Condition C: fine-tuned + Combined SCI (primary result)
!python3 experiment_runner.py --condition C

## Cell 13.5: H5 Data-Scaling Sub-Runs (Condition C with LoRA-2K and LoRA-5K)

**Optional but recommended.** To plot the H5 learning curve (§6.4), Condition C needs to be run with each of the three adapters so we have three (n, PersonaScore) data points. The previous cell already produced the LoRA-10K point; these two extra runs add the LoRA-2K and LoRA-5K points.

**Cost:** ~3 hr A100 + ~$8 in judge calls (2 × 960 probes).

**Why bother:** without these, H5 (the data-scaling hypothesis) is only indirectly supported by training-loss numbers. With them, the analysis script auto-detects the sub-runs and fits a real `a·log(n) + b` curve over PersonaScore.

Logs go to separate folders (`logs/condition_C_lora_2k/` and `logs/condition_C_lora_5k/`) so the primary Condition C run isn't overwritten.

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Condition C with the LoRA-2K adapter (H5 data-scaling point)
!python3 experiment_runner.py --condition C --adapter lora_2k --logs-suffix _lora_2k

# Condition C with the LoRA-5K adapter (H5 data-scaling point)
!python3 experiment_runner.py --condition C --adapter lora_5k --logs-suffix _lora_5k

## Cell 14: Run Analysis

Computes:
- Mean PersonaScore per condition + 95% CI (overall and per-turn time series)
- Per-dimension breakdown (T/E/C/S) for H2 (ΔE = C − D)
- Paired t-test C vs D + Cohen's d for H1 (§6.1)
- 2-way marginal-means analysis (FT × SCI) for H3 (§6.3)
- Failure-mode taxonomy (score ≤ 2 by dimension)
- Decision rules per §7
- Replication-stability check vs Experiment 1 Combined (§6.5)

**H5 learning curve auto-detection:** if you ran Cell 13.5 (Condition C with LoRA-2K and LoRA-5K), the analysis script will automatically detect the sub-runs in `logs/condition_C_lora_*/`, compute mean PersonaScore for each, and plot the `a·log(n) + b` fit. No manual flag needed.

If you want to override with manual data points instead, use:
```bash
!python3 analyse_results.py --learning-curve-data "2000:2.85,5000:3.10,10000:3.32"
```

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 analyse_results.py

## Cell 15: View Results

In [ ]:
from IPython.display import display, Image, Markdown
from pathlib import Path

results_dir = Path(PROJECT_DIR) / 'results'

for plot in ['persona_score_timeseries.png', 'dimension_comparison.png',
             'learning_curve.png', 'condition_comparison.png']:
    p = results_dir / plot
    if p.exists():
        display(Image(filename=str(p), width=800))

report_path = results_dir / 'summary_report.md'
if report_path.exists():
    display(Markdown(report_path.read_text()))

---

## Recovery After Disconnect

1. Re-run **Cell 1** (mount Drive, set API key)
2. Re-run **Cell 2** (install dataset-gen deps) if you're in the data-gen phase
3. Re-run **Cell 7** (install training deps) if you're resuming training, *or* re-run **Cell 11** (verify eval env) if you're in Phase 3 evaluation
4. Re-run the interrupted cell — all training and evaluation runs are resumable from the most recent checkpoint or per-script log

All data + adapters live on Google Drive and survive disconnects.